<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex08.2-transient-heat/Ex08.2_01_soft_ic_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_08.2 · Notebook 01 — Soft Initial Condition

**Paired with L8.2 · Dynamic Heat**

Three competing loss terms:
$\mathcal{L} = \mathcal{L}_{\mathrm{PDE}} + \mathcal{L}_{\mathrm{DBC}}
+ \mathcal{L}_{\mathrm{IC}}$. Watch where the error ends up.

## What you will do

1. Write the parabolic residual, with the time derivative in the right column.
2. Assemble the three-term loss, recording each part separately.
3. Train, then look at *where in time* the error is — not just how big it is.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08.2-transient-heat/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · The problem, the points and the network

3000 points in the slab, the four edges at 20 instants, 400 points on the
initial face. Note the ratio: the initial condition is the only place in this
problem where you were **given** the answer, and it carries about a tenth of
the points.

In [ ]:
C, T_END = 1.0, 1.0
N_F, N_B, N_T, N_0 = 3000, 25, 20, 400

set_seed(88)
model = MLP(n_in=3, n_hidden=40, n_layers=4)
describe(model, N_F)

xyt_f = to_tensor(pb.plate_spacetime_points(N_F, t_end=T_END),
                  requires_grad=True)
xyt_b = to_tensor(boundary_points_in_time(N_B, N_T, pb.PLATE_DOMAIN,
                                          (0.0, T_END), seed=1))
xyt_0 = to_tensor(initial_points(N_0, pb.PLATE_DOMAIN, t0=0.0, seed=1))


def ic_value(xyt):
    """The initial field, sin(pi x) sin(pi y), on the points given."""
    return torch.sin(np.pi * xyt[:, 0:1]) * torch.sin(np.pi * xyt[:, 1:2])

## 2 · The residual (the time derivative is component 2)

$\mathcal{F} = \hat T_{,t} - c\,(\hat T_{,xx} + \hat T_{,yy})$. **One** time
derivative: this is a first-order-in-time equation, which is why one initial
condition determines it.

### Your turn

In [ ]:
# TODO 1 --- the parabolic residual ----------------------------------------------------------
# Two `...` to replace:
#   line 1  ->  grad(T, xyt)[:, 2:3]                        T_t: column 2 of the gradient is time
#   line 2  ->  T_t - C * (d2(T, xyt, 0) + d2(T, xyt, 1))   T_t = c (T_xx + T_yy)
# xyt must have been built with to_tensor(..., requires_grad=True), or grad returns None.
def residual(model, xyt):
    T = model(xyt)
    T_t = ...                                     # <- grad(T, xyt)[:, 2:3]
    return ...                                    # <- T_t - C * (d2(T, xyt, 0) + d2(T, xyt, 1))
# ------------------------------------------------------------------------------

## 3 · The three-term loss, recording each part

Keep the three terms separately as well as summed. A single loss curve tells
you the optimiser is making progress; three curves tell you **which condition
it is making progress on**, and that is the question this notebook asks.

Every quantity here is order one — amplitude 1, $c = 1$, $t_{\mathrm{end}} = 1$
— so the terms need no scaling. That is a property of this benchmark, not a
general licence: check it before you rely on it.

### Your turn

In [ ]:
# TODO 2 --- the three-term loss, term by term -----------------------------------------------
# Three `...` to replace:
#   line 1  ->  mse(residual(model, xyt_f))              the physics
#   line 2  ->  mse(model(xyt_b))                        the edges are at zero
#   line 3  ->  mse(model(xyt_0) - ic_value(xyt_0))      the initial field
parts = {"pde": [], "dbc": [], "ic": []}

def loss_fn():                                    # no arguments: it closes over model and the points
    L_pde = ...                                   # <- mse(residual(model, xyt_f))
    L_dbc = ...                                   # <- mse(model(xyt_b))
    L_ic  = ...                                   # <- mse(model(xyt_0) - ic_value(xyt_0))
    for k, v in zip(("pde", "dbc", "ic"), (L_pde, L_dbc, L_ic)):
        parts[k].append(v.item())
    return L_pde + L_dbc + L_ic
# ------------------------------------------------------------------------------

## 4 · Train

Adam to get near a solution, L-BFGS to polish. L-BFGS is full-batch by
construction and calls the loss several times per step during its line search,
so the points must not change once training has started.

In [ ]:
history = train_two_stage(model, loss_fn, adam_steps=4000, lbfgs_steps=200,
                          lr=1e-3)

fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.2))
plot_curves(history, ax=axes[0], title="the plate — three soft terms")
for i, k in enumerate(("pde", "dbc", "ic")):
    axes[1].semilogy(parts[k], lw=1.4, color=CYCLE[i], label=k)
axes[1].set_xlabel("loss evaluation"); axes[1].set_ylabel("term")
axes[1].set_title("...and which term is which")
axes[1].legend(frameon=False, fontsize=9); axes[1].grid(alpha=0.25, which="both")
plt.tight_layout(); plt.show()

## 5 · Where in time is the error?

The usual report is one number for the whole slab. That hides the thing worth
knowing: a transient problem is not uniformly hard.

In [ ]:
ts, rel, ab = pb.error_vs_time(model, c=C)

print(error_table(
    [[f"{t:.1f}", f"{r:.3e}", f"{a:.3e}"] for t, r, a in zip(ts, rel, ab)],
    ["t", "relative L2", "max absolute"]))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].semilogy(ts, rel, "o-", color="#1f77b4")
ax[0].set_title("relative"); ax[0].set_xlabel("t"); ax[0].grid(alpha=0.25, which="both")
ax[1].semilogy(ts, ab, "o-", color="#d94f2b")
ax[1].set_title("absolute"); ax[1].set_xlabel("t"); ax[1].grid(alpha=0.25, which="both")
plt.tight_layout(); plt.show()

**Question.** Which instant has the largest error, and why is it that one?

Read the two panels together before you answer. The relative and the absolute
curves do not have to agree, and where they disagree the reason is in the last
column of notebook 00's table.

---

## 6 · What the model actually produced

In [ ]:
show = [0.0, 0.05, 0.2]
fig, axes = plt.subplots(2, len(show), figsize=(13.0, 7.0))
for j, t in enumerate(show):
    X, Y, U = pb.slice_at_time(model, t)
    E = pb.exact_transient(X, Y, t, C)
    pb.plot_slice(X, Y, U, ax=axes[0, j], title=f"model, t = {t}", label="T")
    pb.plot_slice(X, Y, U - E, ax=axes[1, j], title=f"error, t = {t}",
                  label="T", cmap="coolwarm")
plt.tight_layout(); plt.show()

## 7 · Save

In [ ]:
os.makedirs("Ex08.2_outputs", exist_ok=True)
path = os.path.join("Ex08.2_outputs", "nb01_soft.npz")
np.savez(path, ts=ts, rel=rel, abs_err=ab,
         adam=history["adam"], lbfgs=history["lbfgs"],
         pde=np.asarray(parts["pde"]), dbc=np.asarray(parts["dbc"]),
         ic=np.asarray(parts["ic"]))
torch.save(model.state_dict(), os.path.join("Ex08.2_outputs", "nb01_soft.pt"))
print("wrote", path)

## 8 · Before you move on

1. The initial condition is one term among three, and the interior carries
   most of the points. What does that do to the balance between the terms?
2. You recorded the three terms separately. Which one fell first, which one
   plateaued, and what does that ordering tell you about the optimiser's
   priorities?
3. The relative error at late times is large and the absolute error is tiny.
   Say which of the two you would put in a report, and to whom.

Next: **notebook 02**, where the initial condition is built into the trial
solution and cannot be traded away.